# 04 — Mode-aware ExGAN

Builds on `03` (ExGAN-noshift = conditional WGAN-GP + EVT/GPD conditioning on LOS), which was the better ExGAN variant.
`03` showed the GAN generates patients that are too "empty" (unexplained 0.34 vs 0.18 real) and under-produces rare modes.

**Changes in this notebook**
1. **Mode conditioning:** generator and critic also receive the 3 mode flags (addiction_pain, cardiopulmonary, oncology_transplant; all-zero = unexplained).
2. **Mode consistency loss:** the generator is penalised (BCE) when its output does not contain the requested modes (CTGAN-style conditional loss).
3. **Training-by-sampling:** batches are drawn so each mode pattern appears with probability ∝ log(frequency + 1), so rare patterns are seen often (CTGAN's trick).
4. **Sampling:** mode patterns are bootstrapped from the real PLOS rows, LOS is drawn from the GPD (same EVT step as `03`). No distribution shifting.

**Fair evaluation.** Mode shares are now partly controlled by construction, so the key metrics are ones the conditioning does *not* control:
- `nonmode_prev_err` / `nonmode_corr_err`: condition prevalence and correlations on all flags **outside** the mode definitions
- `within_mode_err`: inside each mode, do synthetic patients have the same *other* conditions as real patients in that mode?
- `mode_consistency`: share of rows whose generated modes match the requested pattern (does the conditioning work?)

All 7 generators (02, 03, 04) are scored on these.


In [ ]:
# Cell 1 — paths and settings
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/genai_synthea/run_01"
OUT  = f"{BASE}/genai"
SYN_DIR = f"{OUT}/synthetic"
RES_DIR = f"{OUT}/results"

SEEDS  = (42,)          # use (42, 43, 44) for final numbers
N_SYN  = 1000
EPOCHS = 1000           # = INIT_EPOCHS + k*SHIFT_EPOCHS in 03, so compute matches ExGAN-noshift
BATCH, Z_DIM, HIDDEN = 128, 64, 256
N_CRITIC, GP_W, LR = 5, 10.0, 2e-4
GUMBEL_TAU = 0.2
LAMBDA_MODE = 1.0       # weight of the mode consistency loss


In [ ]:
# Cell 2 — load data, modes
import os, json, time, math, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.stats import genpareto
warnings.filterwarnings("ignore")
os.makedirs(SYN_DIR, exist_ok=True); os.makedirs(RES_DIR, exist_ok=True)

meta  = json.load(open(f"{OUT}/meta.json"))
train = pd.read_parquet(f"{OUT}/train.parquet")
val   = pd.read_parquet(f"{OUT}/val.parquet")
THR, MODES = meta["plos_threshold"], meta["modes"]
CAT, BIN   = meta["cat_cols"], meta["bin_cols"]
NUM = meta["num_cols"] + ["target"]
GEN_COLS = NUM + CAT + BIN
MODE_NAMES = list(MODES.keys())
MODE_FLAGS = sorted({c for v in MODES.values() for c in v if c in BIN})
NONMODE = [c for c in BIN if c not in MODE_FLAGS]

DEV = "cuda" if torch.cuda.is_available() else "cpu"
real_tail = train[train["is_plos"] == 1][GEN_COLS].reset_index(drop=True)

def mode_matrix(d):
    return np.stack([d[[c for c in MODES[k] if c in d.columns]].max(axis=1).values for k in MODE_NAMES], 1).astype(np.float32)

M_train = mode_matrix(train)
patterns = ["".join(map(str, r.astype(int))) for r in M_train]
print(f"Device {DEV} | train {len(train)} | PLOS {len(real_tail)} | mode flags {len(MODE_FLAGS)} | other flags {len(NONMODE)}")
print("Mode patterns in train (addiction, cardio, onc):")
print(pd.Series(patterns).value_counts().to_string())


In [ ]:
# Cell 3 — (same as 03) encoder / decoder (age z-score, one-hot categoricals, 2-way one-hot per condition flag)
class Codec:
    def __init__(self, df):
        self.age_mu, self.age_sd = df["anchor_age"].mean(), df["anchor_age"].std()
        self.age_min, self.age_max = df["anchor_age"].min(), df["anchor_age"].max()
        logl = np.log(df["target"].clip(lower=1))
        self.l_mu, self.l_sd = logl.mean(), logl.std()
        self.levels = {c: sorted(df[c].astype(str).unique()) for c in CAT}
        self.cat_sizes = [len(self.levels[c]) for c in CAT]
        self.n_bin = len(BIN)
        self.dim = 1 + sum(self.cat_sizes) + 2 * self.n_bin

    def encode_x(self, df):
        parts = [((df["anchor_age"].values - self.age_mu) / self.age_sd)[:, None]]
        for c in CAT:
            v = df[c].astype(str).values
            parts.append(np.stack([(v == l) for l in self.levels[c]], 1).astype(np.float32))
        b = df[BIN].values.astype(np.float32)
        parts.append(np.stack([1 - b, b], 2).reshape(len(df), -1))
        return np.concatenate(parts, 1).astype(np.float32)

    def encode_e(self, los):
        return ((np.log(np.clip(los, 1, None)) - self.l_mu) / self.l_sd).astype(np.float32)[:, None]

    def decode_e(self, e):
        return np.exp(e.ravel() * self.l_sd + self.l_mu)

    def decode(self, x, los):
        out, i = {}, 1
        out["anchor_age"] = np.clip(np.round(x[:, 0] * self.age_sd + self.age_mu), self.age_min, self.age_max)
        out["target"] = np.round(los)
        for c, s in zip(CAT, self.cat_sizes):
            out[c] = np.array(self.levels[c])[x[:, i:i + s].argmax(1)]; i += s
        flags = x[:, i:].reshape(len(x), self.n_bin, 2).argmax(2)
        df = pd.DataFrame(out)
        df = pd.concat([df, pd.DataFrame(flags, columns=BIN)], axis=1)
        return df[GEN_COLS]

codec = Codec(train)
print("Encoded row dim:", codec.dim)


In [ ]:
# Cell 4 — mode-conditioned networks, consistency loss, training-by-sampling
NM = len(MODE_NAMES)
bin_start = 1 + sum(codec.cat_sizes)
mode_bin_idx = [[BIN.index(c) for c in MODES[k] if c in BIN] for k in MODE_NAMES]

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(Z_DIM + 1 + NM, HIDDEN), nn.ReLU(),
                                 nn.Linear(HIDDEN, HIDDEN), nn.ReLU(),
                                 nn.Linear(HIDDEN, codec.dim))
    def forward(self, z, e, m):
        h = self.net(torch.cat([z, e, m], 1))
        outs, i = [h[:, :1]], 1
        for s in codec.cat_sizes:
            outs.append(F.gumbel_softmax(h[:, i:i + s], tau=GUMBEL_TAU)); i += s
        b = h[:, i:].reshape(-1, codec.n_bin, 2)
        outs.append(F.gumbel_softmax(b, tau=GUMBEL_TAU, dim=2).reshape(len(h), -1))
        return torch.cat(outs, 1)

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(codec.dim + 1 + NM, HIDDEN), nn.LeakyReLU(0.2),
                                 nn.Linear(HIDDEN, HIDDEN), nn.LeakyReLU(0.2), nn.Linear(HIDDEN, 1))
    def forward(self, x, e, m):
        return self.net(torch.cat([x, e, m], 1))

def mode_probs(x):
    """Differentiable 'has mode k' = max over the mode's flags of P(flag = 1)."""
    p1 = x[:, bin_start + 1::2]            # P(flag=1) for every condition flag
    return torch.stack([p1[:, idx].max(1).values for idx in mode_bin_idx], 1)

def grad_penalty(D, real, fake, e, m):
    a = torch.rand(len(real), 1, device=DEV)
    mix = (a * real + (1 - a) * fake).requires_grad_(True)
    g = torch.autograd.grad(D(mix, e, m).sum(), mix, create_graph=True)[0]
    return ((g.norm(2, dim=1) - 1) ** 2).mean()

# training-by-sampling: P(pattern) ∝ log(freq+1), uniform within pattern
pat_s = pd.Series(patterns)
freq = pat_s.map(pat_s.value_counts()).values.astype(float)
row_w = np.log(freq + 1) / freq
row_w = torch.tensor(row_w / row_w.sum(), dtype=torch.float)

def train_ma(G, D, X, E, M, epochs, optG, optD, log_every=200):
    X, E, M = (torch.tensor(a, device=DEV) for a in (X, E, M))
    n = len(X); steps = math.ceil(n / BATCH)
    for ep in range(1, epochs + 1):
        for _ in range(steps):
            idx = torch.multinomial(row_w, BATCH, replacement=True).to(DEV)
            real, e, m = X[idx], E[idx], M[idx]
            for _ in range(N_CRITIC):
                fake = G(torch.randn(BATCH, Z_DIM, device=DEV), e, m).detach()
                lossD = D(fake, e, m).mean() - D(real, e, m).mean() + GP_W * grad_penalty(D, real, fake, e, m)
                optD.zero_grad(); lossD.backward(); optD.step()
            fake = G(torch.randn(BATCH, Z_DIM, device=DEV), e, m)
            l_mode = F.binary_cross_entropy(mode_probs(fake).clamp(1e-6, 1 - 1e-6), m)
            lossG = -D(fake, e, m).mean() + LAMBDA_MODE * l_mode
            optG.zero_grad(); lossG.backward(); optG.step()
        if ep % log_every == 0 or ep == epochs:
            print(f"    epoch {ep}/{epochs} | lossD {lossD.item():.3f} | mode BCE {l_mode.item():.3f}")

@torch.no_grad()
def generate_ma(G, los, m):
    G.eval()
    e = torch.tensor(codec.encode_e(np.asarray(los, float)), device=DEV)
    m = torch.tensor(np.asarray(m, np.float32), device=DEV)
    x = G(torch.randn(len(e), Z_DIM, device=DEV), e, m).cpu().numpy()
    G.train(); return x


In [ ]:
# Cell 5 — EVT step (same as 03): GPD on real PLOS LOS above the threshold
exceed = real_tail["target"].values - THR
xi, _, sigma = genpareto.fit(exceed, floc=0)
LOS_CAP = 1.2 * real_tail["target"].max()
print(f"GPD: xi = {xi:.3f}, sigma = {sigma:.2f}")
def sample_los_gpd(n, rng):
    los = THR + genpareto.rvs(xi, loc=0, scale=sigma, size=n, random_state=rng)
    return np.clip(np.maximum(np.round(los), math.floor(THR) + 1), None, LOS_CAP)
def los_at_tail_prob(tau):
    return THR + genpareto.ppf(1 - tau, xi, loc=0, scale=sigma)
M_tail = mode_matrix(real_tail)


In [ ]:
# Cell 6 — train mode-aware ExGAN (one model per seed), sample PLOS rows
results_syn, fit_time, models, consistency = {}, {}, {}, {}
X_all = codec.encode_x(train[GEN_COLS]); E_all = codec.encode_e(train["target"].values.astype(float))
for seed in SEEDS:
    name = "ExGAN-MA" if len(SEEDS) == 1 else f"ExGAN-MA_s{seed}"
    torch.manual_seed(seed); np.random.seed(seed); rng = np.random.default_rng(seed)
    G, D = Generator().to(DEV), Critic().to(DEV)
    optG = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.9))
    optD = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.9))
    print(f"Training {name} ...")
    t0 = time.time()
    train_ma(G, D, X_all, E_all, M_train, EPOCHS, optG, optD)
    m_req = M_tail[rng.integers(0, len(M_tail), N_SYN)]          # bootstrap real PLOS mode patterns
    e = sample_los_gpd(N_SYN, rng)
    syn = codec.decode(generate_ma(G, e, m_req), e)
    consistency[name] = float((mode_matrix(syn) == m_req).all(1).mean())
    results_syn[name], fit_time[name], models[name] = syn, time.time() - t0, G
    syn.to_parquet(f"{SYN_DIR}/{name}.parquet", index=False)
    print(f"{name}: {len(syn)} rows | mode consistency {consistency[name]:.1%} | {fit_time[name]:.0f}s")


In [ ]:
# Cell 7 — score ALL generators (02 + 03 + 04), including the metrics conditioning cannot game
from scipy.stats import wasserstein_distance, ks_2samp
from sklearn.neighbors import NearestNeighbors

synthetic = {}
for name in ["SMOTE-NC", "GaussianCopula", "CTGAN", "TVAE", "ExGAN", "ExGAN-noshift"]:
    p = f"{SYN_DIR}/{name}.parquet"
    if os.path.exists(p): synthetic[name] = pd.read_parquet(p)
synthetic.update(results_syn)
prev_fit = {}
for f_ in ["02_fidelity.csv", "03_fidelity_all.csv"]:
    if os.path.exists(f"{RES_DIR}/{f_}"):
        prev_fit.update(pd.read_csv(f"{RES_DIR}/{f_}", index_col=0)["fit_s"].dropna().to_dict())

def mode_flags(d):
    f = pd.DataFrame(mode_matrix(d), columns=MODE_NAMES, index=d.index).astype(int)
    f["unexplained"] = (f.sum(axis=1) == 0).astype(int)
    return f

real_flags = mode_flags(real_tail).mean()
real_p90 = real_tail["target"].quantile(0.90)
age_mu, age_sd = real_tail["anchor_age"].mean(), real_tail["anchor_age"].std()
los_mu, los_sd = real_tail["target"].mean(), real_tail["target"].std()
cat_levels = {c: sorted(real_tail[c].astype(str).unique()) for c in CAT}
def encode(d):
    parts = [((d["anchor_age"] - age_mu) / age_sd).to_frame(), ((d["target"] - los_mu) / los_sd).to_frame(), d[BIN].astype(float)]
    for c in CAT:
        parts.append(pd.DataFrame({f"{c}={l}": (d[c].astype(str) == l).astype(float) for l in cat_levels[c]}, index=d.index))
    return pd.concat(parts, axis=1).values
nn_ = NearestNeighbors(n_neighbors=1).fit(encode(real_tail))
holdout = val[val["is_plos"] == 1][GEN_COLS].reset_index(drop=True)
dcr_holdout = np.median(nn_.kneighbors(encode(holdout))[0])

def corr_err(a, b, cols):
    ca, cb = a[cols].corr().fillna(0).values, b[cols].corr().fillna(0).values
    d = np.abs(ca - cb); return d[np.triu_indices_from(d, 1)].mean()

real_mf = mode_flags(real_tail)
def within_mode_err(s):
    sf, errs = mode_flags(s), []
    for k in MODE_NAMES:
        r, g = real_tail[real_mf[k] == 1], s[sf[k] == 1]
        if len(r) >= 20 and len(g) >= 20:
            errs.append((r[NONMODE].mean() - g[NONMODE].mean()).abs().mean())
    return np.mean(errs) if errs else np.nan

rows = []
for name, s in synthetic.items():
    s = s[GEN_COLS].copy()
    for c in BIN: s[c] = s[c].astype(int)
    f = mode_flags(s).mean()
    d_syn = nn_.kneighbors(encode(s))[0].ravel()
    rows.append({"generator": name,
        "LOS_KS": ks_2samp(real_tail["target"], s["target"]).statistic,
        "extreme_cov": (s["target"] > real_p90).mean(), "LOS_max": s["target"].max(),
        **{f"mode_{k}": f[k] for k in f.index},
        "mode_abs_err": (f - real_flags).abs().mean(),
        "nonmode_prev_err": (s[NONMODE].mean() - real_tail[NONMODE].mean()).abs().mean(),
        "nonmode_corr_err": corr_err(s, real_tail, ["anchor_age", "target"] + NONMODE),
        "within_mode_err": within_mode_err(s),
        "mode_consistency": consistency.get(name, np.nan),
        "exact_copies": (d_syn < 1e-6).mean(), "DCR_ratio": np.median(d_syn) / dcr_holdout,
        "fit_s": fit_time.get(name, prev_fit.get(name, np.nan))})

real_row = {"generator": "REAL (train PLOS)", "LOS_KS": 0, "extreme_cov": (real_tail["target"] > real_p90).mean(),
            "LOS_max": real_tail["target"].max(), **{f"mode_{k}": real_flags[k] for k in real_flags.index}}
fid = pd.DataFrame([real_row] + rows).set_index("generator")
fid.to_csv(f"{RES_DIR}/04_fidelity_all.csv")
print(f"Real PLOS {len(real_tail)} | mode-defining flags {len(MODE_FLAGS)} | other flags {len(NONMODE)} | holdout DCR {dcr_holdout:.2f}")
fid.round(3).T


In [ ]:
# Cell 8 — scorecard (targets from 02, plus the non-gameable checks)
ref = fid.drop(index="REAL (train PLOS)")
score = pd.DataFrame({
    "modes_ok (err<=0.06)":    ref["mode_abs_err"] <= 0.06,
    "extremes_ok (cov>=0.08)": ref["extreme_cov"] >= 0.08,
    "max_ok (>=90 d)":         ref["LOS_max"] >= 90,
    "no_copying (DCR>=1)":     ref["DCR_ratio"] >= 1.0,
    "nonmode_best3":           ref["nonmode_prev_err"].rank() <= 3,
    "within_mode_best3":       ref["within_mode_err"].rank() <= 3,
})
score["targets_met"] = score.sum(axis=1)
print(score.sort_values("targets_met", ascending=False).to_string())


In [ ]:
# Cell 9 — plots
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
mc = [c for c in fid.columns if c.startswith("mode_") and c not in ("mode_abs_err", "mode_consistency")]
fid[mc].rename(columns=lambda c: c.replace("mode_", "")).T.plot.bar(ax=axes[0], rot=0)
axes[0].set_ylabel("share of PLOS rows"); axes[0].set_title("Mode coverage"); axes[0].legend(fontsize=7)
ref[["nonmode_prev_err", "within_mode_err"]].plot.barh(ax=axes[1])
axes[1].set_title("Errors the conditioning cannot control (lower = better)"); axes[1].invert_yaxis()
fig.tight_layout(); fig.savefig(f"{RES_DIR}/04_plots.png", dpi=150); plt.show()


In [ ]:
# Cell 10 — targeted generation: rare-mode patients at a chosen extremeness
G = models[list(models)[0]]
for k_i, k in enumerate(MODE_NAMES):
    for tau in [0.10, 0.01]:
        L = los_at_tail_prob(tau); m = np.zeros((200, NM), np.float32); m[:, k_i] = 1
        rows_ = codec.decode(generate_ma(G, np.full(200, L), m), np.full(200, round(L)))
        hit = mode_flags(rows_)[k].mean()
        print(f"{k:20s} | 1-in-{int(1/tau):<3d} (LOS ~ {L:.0f} d) | requested mode present in {hit:.0%} | mean age {rows_['anchor_age'].mean():.0f}")
